# Modelisation From Scratch — Reseau de Hubs Standard

**Auteur :** Athanor SAVOUILLAN  
**Date :** Mars 2026  
**Approche :** K-Means from scratch, hubs Standard uniquement (rayon 80 km)

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import folium
from folium.plugins import HeatMap
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from pathlib import Path
from math import radians, sin, cos, sqrt, atan2

# --- Constantes metier ---
MAX_RADIUS_KM: float = 80.0
CAPEX_STANDARD: int = 2_000_000       # USD
OPEX_STANDARD: int = 311_000          # USD / an
DRONES_MIN: int = 10
DRONES_MAX: int = 20
RANDOM_STATE: int = 42
URBAN_TYPES: set[str] = {
    "city", "suburb", "neighbourhood"
}

EARTH_R: float = 6_371.0


def haversine_km(
    lat1: float,
    lon1: float,
    lat2: float,
    lon2: float,
) -> float:
    """Distance Haversine vectorisee (km)."""
    lat1 = np.radians(lat1)
    lat2 = np.radians(lat2)
    lon1 = np.radians(lon1)
    lon2 = np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )
    return EARTH_R * 2 * np.arctan2(
        np.sqrt(a), np.sqrt(1 - a)
    )


# --- Chargement des donnees ---
DATA_DIR: Path = (
    Path("..") / ".." / "data" / "Final Group"
)

df_facilities: pd.DataFrame = pd.read_csv(
    DATA_DIR / "ghana_health_eda_final.csv"
)
df_villages: pd.DataFrame = pd.read_csv(
    DATA_DIR / "ghana_villages_eda_final.csv"
)

print(
    f"Facilities : {len(df_facilities)} lignes"
)
print(
    f"Villages   : {len(df_villages)} lignes"
)

In [ ]:
# Villages ruraux = hors types urbains
df_rural: pd.DataFrame = df_villages[
    ~df_villages["Place_Type"].isin(URBAN_TYPES)
].copy()

n_rural: int = len(df_rural)
n_total: int = len(df_villages)
pct_rural: float = n_rural / n_total * 100

print(f"Villages ruraux : {n_rural} / {n_total}")
print(f"Soit {pct_rural:.1f}% du total")

# Facilities desservant au moins 1 village rural
rural_fac_names: set = set(
    df_rural["Nearest_Facility_Name"].dropna()
)
df_fac_rural: pd.DataFrame = df_facilities[
    df_facilities["Facility_Name"]
    .isin(rural_fac_names)
].copy()

print(
    f"Facilities rurales : {len(df_fac_rural)}"
    f" / {len(df_facilities)}"
)

Le filtrage exclut les zones urbaines (city, suburb, neighbourhood) ou les infrastructures de sante sont deja denses. Le reseau de drones cible les villages ruraux eloignes.

In [ ]:
fac_coords: np.ndarray = df_fac_rural[
    ["Latitude", "Longitude"]
].values

k_range: range = range(5, 21)
results: list[dict] = []

for k in k_range:
    model = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=10,
    )
    labels = model.fit_predict(fac_coords)
    centers = model.cluster_centers_

    # Couverture Haversine
    dists = np.array([
        haversine_km(
            fac_coords[i, 0],
            fac_coords[i, 1],
            centers[labels[i], 0],
            centers[labels[i], 1],
        )
        for i in range(len(fac_coords))
    ])
    covered = np.sum(dists <= MAX_RADIUS_KM)
    coverage = covered / len(fac_coords) * 100

    sil = silhouette_score(
        fac_coords, labels
    )
    inertia = model.inertia_

    # TCO 5 ans
    tco = k * (
        CAPEX_STANDARD + 5 * OPEX_STANDARD
    ) / 1_000_000

    results.append({
        "K": k,
        "Silhouette": round(sil, 4),
        "Inertie": round(inertia, 1),
        "Couverture_pct": round(coverage, 1),
        "TCO_5ans_M": round(tco, 1),
    })

df_results: pd.DataFrame = pd.DataFrame(results)
df_results

In [ ]:
fig_dual = make_subplots(
    specs=[[{"secondary_y": True}]],
)

fig_dual.add_trace(
    go.Scatter(
        x=df_results["K"],
        y=df_results["Couverture_pct"],
        name="Couverture (%)",
        mode="lines+markers",
        line=dict(color="#00D4AA", width=2),
        marker=dict(size=8),
    ),
    secondary_y=False,
)

fig_dual.add_trace(
    go.Scatter(
        x=df_results["K"],
        y=df_results["TCO_5ans_M"],
        name="TCO 5 ans (M$)",
        mode="lines+markers",
        line=dict(
            color="#FF4757",
            width=2,
            dash="dash",
        ),
        marker=dict(size=8),
    ),
    secondary_y=True,
)

fig_dual.update_layout(
    title="Couverture vs TCO selon K",
    template="plotly_dark",
    xaxis_title="Nombre de hubs (K)",
    height=450,
)
fig_dual.update_yaxes(
    title_text="Couverture (%)",
    secondary_y=False,
)
fig_dual.update_yaxes(
    title_text="TCO 5 ans (M$)",
    secondary_y=True,
)
fig_dual.show()

### Choix des scenarios

- **K_A** = premier K depassant 90% de couverture (objectif cout optimal)  
- **K_B** = K avec la couverture maximale  

Les deux scenarios utilisent exclusivement des hubs Standard (80 km, 2 M$ CAPEX).

---

## Scenario A — Cout optimal (> 90%)

In [ ]:
# Identification K_A et K_B
mask_90 = df_results["Couverture_pct"] > 90
K_A: int = int(
    df_results.loc[mask_90, "K"].iloc[0]
) if mask_90.any() else 15

idx_max = df_results["Couverture_pct"].idxmax()
K_B: int = int(df_results.loc[idx_max, "K"])

print(f"K_A (premier > 90%) : {K_A}")
print(f"K_B (couverture max) : {K_B}")

# --- Scenario A ---
model_a = KMeans(
    n_clusters=K_A,
    random_state=RANDOM_STATE,
    n_init=10,
)
labels_a = model_a.fit_predict(fac_coords)
centers_a = model_a.cluster_centers_

hub_data_a: list[dict] = []
for c in range(K_A):
    mask_c = labels_a == c
    n_fac = int(mask_c.sum())
    dists_c = haversine_km(
        fac_coords[mask_c, 0],
        fac_coords[mask_c, 1],
        centers_a[c, 0],
        centers_a[c, 1],
    )
    covered = int(
        np.sum(dists_c <= MAX_RADIUS_KM)
    )
    drones = max(
        DRONES_MIN,
        min(DRONES_MAX, n_fac // 20),
    )
    hub_data_a.append({
        "Hub": f"A-{c+1:02d}",
        "Lat": round(centers_a[c, 0], 5),
        "Lon": round(centers_a[c, 1], 5),
        "Facilities": n_fac,
        "Covered": covered,
        "Drones": drones,
        "Max_dist_km": round(
            float(dists_c.max()), 1
        ),
    })

df_hubs_a: pd.DataFrame = pd.DataFrame(
    hub_data_a
)

cov_a: float = (
    df_hubs_a["Covered"].sum()
    / len(fac_coords) * 100
)
capex_a: float = K_A * CAPEX_STANDARD / 1e6
tco_a: float = K_A * (
    CAPEX_STANDARD + 5 * OPEX_STANDARD
) / 1e6

print(f"Scenario A : K={K_A}")
print(f"  Couverture : {cov_a:.1f}%")
print(f"  CAPEX : {capex_a:.1f} M$")
print(f"  TCO 5 ans : {tco_a:.1f} M$")
print()
df_hubs_a

In [ ]:
# Carte Scenario A
m_a = folium.Map(
    location=[7.95, -1.03],
    zoom_start=7,
    tiles="CartoDB positron",
)

# Cercles de couverture 80 km (dessines en premier)
for _, row in df_hubs_a.iterrows():
    folium.Circle(
        location=[row["Lat"], row["Lon"]],
        radius=MAX_RADIUS_KM * 1000,
        color="#FFA502",
        fill=True,
        fill_opacity=0.08,
        weight=1.5,
    ).add_to(m_a)

# Facilities rurales (vert = couvertes, rouge = non couvertes)
for idx in range(len(df_fac_rural)):
    flat = df_fac_rural.iloc[idx]["Latitude"]
    flon = df_fac_rural.iloc[idx]["Longitude"]
    # Verifier si couverte par le hub assigne
    hub_id = labels_a[idx]
    c_lat, c_lon = centers_a[hub_id]
    dist = haversine_km(flat, flon, c_lat, c_lon)
    color = "#2ecc71" if dist <= MAX_RADIUS_KM else "#e74c3c"
    folium.CircleMarker(
        location=[flat, flon],
        radius=2,
        color=color,
        fill=True,
        fill_opacity=0.7,
        weight=0.5,
    ).add_to(m_a)

# Marqueurs hubs (icone avion noir)
for _, row in df_hubs_a.iterrows():
    folium.Marker(
        location=[row["Lat"], row["Lon"]],
        popup=(
            f"Hub {row['Hub']} (Standard)<br>"
            f"Facilities : {row['Facilities']}<br>"
            f"Drones : {row['Drones']}"
        ),
        icon=folium.Icon(
            color="black", icon="plane", prefix="fa",
        ),
    ).add_to(m_a)

# Legende

m_a

Le scenario A atteint l'objectif de 90% avec le nombre minimal de hubs Standard.

---

## Scenario B — Couverture maximale

In [ ]:
# --- Scenario B ---
model_b = KMeans(
    n_clusters=K_B,
    random_state=RANDOM_STATE,
    n_init=10,
)
labels_b = model_b.fit_predict(fac_coords)
centers_b = model_b.cluster_centers_

hub_data_b: list[dict] = []
for c in range(K_B):
    mask_c = labels_b == c
    n_fac = int(mask_c.sum())
    dists_c = haversine_km(
        fac_coords[mask_c, 0],
        fac_coords[mask_c, 1],
        centers_b[c, 0],
        centers_b[c, 1],
    )
    covered = int(
        np.sum(dists_c <= MAX_RADIUS_KM)
    )
    drones = max(
        DRONES_MIN,
        min(DRONES_MAX, n_fac // 20),
    )
    hub_data_b.append({
        "Hub": f"B-{c+1:02d}",
        "Lat": round(centers_b[c, 0], 5),
        "Lon": round(centers_b[c, 1], 5),
        "Facilities": n_fac,
        "Covered": covered,
        "Drones": drones,
        "Max_dist_km": round(
            float(dists_c.max()), 1
        ),
    })

df_hubs_b: pd.DataFrame = pd.DataFrame(
    hub_data_b
)

cov_b: float = (
    df_hubs_b["Covered"].sum()
    / len(fac_coords) * 100
)
capex_b: float = K_B * CAPEX_STANDARD / 1e6
tco_b: float = K_B * (
    CAPEX_STANDARD + 5 * OPEX_STANDARD
) / 1e6

print(f"Scenario B : K={K_B}")
print(f"  Couverture : {cov_b:.1f}%")
print(f"  CAPEX : {capex_b:.1f} M$")
print(f"  TCO 5 ans : {tco_b:.1f} M$")
print()
df_hubs_b

In [ ]:
# Carte Scenario B
m_b = folium.Map(
    location=[7.95, -1.03],
    zoom_start=7,
    tiles="CartoDB positron",
)

# Cercles de couverture 80 km
for _, row in df_hubs_b.iterrows():
    folium.Circle(
        location=[row["Lat"], row["Lon"]],
        radius=MAX_RADIUS_KM * 1000,
        color="#4FC3F7",
        fill=True,
        fill_opacity=0.08,
        weight=1.5,
    ).add_to(m_b)

# Facilities rurales
for idx in range(len(df_fac_rural)):
    flat = df_fac_rural.iloc[idx]["Latitude"]
    flon = df_fac_rural.iloc[idx]["Longitude"]
    hub_id = labels_b[idx]
    c_lat, c_lon = centers_b[hub_id]
    dist = haversine_km(flat, flon, c_lat, c_lon)
    color = "#2ecc71" if dist <= MAX_RADIUS_KM else "#e74c3c"
    folium.CircleMarker(
        location=[flat, flon],
        radius=2,
        color=color,
        fill=True,
        fill_opacity=0.7,
        weight=0.5,
    ).add_to(m_b)

# Marqueurs hubs
for _, row in df_hubs_b.iterrows():
    folium.Marker(
        location=[row["Lat"], row["Lon"]],
        popup=(
            f"Hub {row['Hub']} (Standard)<br>"
            f"Facilities : {row['Facilities']}<br>"
            f"Drones : {row['Drones']}"
        ),
        icon=folium.Icon(
            color="blue", icon="plane", prefix="fa",
        ),
    ).add_to(m_b)

# Legende

m_b

Le scenario B maximise la couverture au prix d'un budget plus eleve.

In [ ]:
# Tableau comparatif
# Couverture Zipline = 73% (reference BA)
df_compare: pd.DataFrame = pd.DataFrame([
    {
        "Scenario": "Zipline (actuel)",
        "Hubs": 4,
        "Type": "Large",
        "Couverture_pct": 73.0,
        "CAPEX_M": None,
        "TCO_5ans_M": None,
    },
    {
        "Scenario": f"A — Cout optimal (K={K_A})",
        "Hubs": K_A,
        "Type": "Standard",
        "Couverture_pct": round(cov_a, 1),
        "CAPEX_M": round(capex_a, 1),
        "TCO_5ans_M": round(tco_a, 1),
    },
    {
        "Scenario": f"B — Couv. max (K={K_B})",
        "Hubs": K_B,
        "Type": "Standard",
        "Couverture_pct": round(cov_b, 1),
        "CAPEX_M": round(capex_b, 1),
        "TCO_5ans_M": round(tco_b, 1),
    },
])

df_compare

In [ ]:
fig_bar = px.bar(
    df_compare,
    x="Scenario",
    y="Couverture_pct",
    color="Scenario",
    color_discrete_sequence=[
        "#FF9800", "#00D4AA", "#4FC3F7",
    ],
    text="Couverture_pct",
    title=(
        "Couverture des facilities "
        "par scenario"
    ),
    template="plotly_dark",
)
fig_bar.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside",
)
fig_bar.update_layout(
    yaxis_title="Couverture (%)",
    showlegend=False,
    height=400,
)
fig_bar.show()

### Synthese

Le reseau Zipline actuel couvre 73% des facilities avec 4 hubs Large. Le scenario A depasse 90% avec des hubs Standard, pour un investissement maitrise. Le scenario B pousse la couverture au maximum, a arbitrer selon le budget disponible.

In [ ]:
# Export des resultats
out_dir: Path = (
    Path("..") / ".." / "data" / "processed"
)
out_dir.mkdir(parents=True, exist_ok=True)

df_hubs_a.to_csv(
    out_dir / "hubs_athanor_standard.csv",
    index=False,
)
df_results.to_csv(
    out_dir / "k_search_athanor.csv",
    index=False,
)

print("Exports :")
print(
    f"  {out_dir / 'hubs_athanor_standard.csv'}"
)
print(
    f"  {out_dir / 'k_search_athanor.csv'}"
)